# Paper 001: AAD cloud experiment workflow

This Colab-first notebook runs the repository's existing command-line
modules and also remains usable in local Jupyter or Kaggle. It does **not**
contain another training implementation. Edit the parameter cell first,
then run only the stages you
need.

All costly, network, archive, and test actions are disabled by default. Model
selection must use validation evidence. Keep the final test stage locked until
the architecture and checkpoint are frozen.

## Recommended order

1. Report the environment and validate paths.
2. Preview dataset samples and smoke-test the model.
3. Inspect the committed protocol and candidate list.
4. Run architecture screening or baseline confirmation only when required.
5. Train the chosen custom model.
6. Inspect validation artifacts.
7. Explicitly unlock one final test evaluation.
8. Archive the chosen run to persistent storage.

The original screen used 16 frames at 32×32. The extended-training defaults
below use 32 frames at 64×64. Those results are a new protocol and must not be
reported as an architecture-only comparison with the original screen.

In [ ]:
from pathlib import Path

# Colab-first repository, dataset, and temporary output locations.
REPOSITORY_URL = "https://github.com/sanelehlabisa/phd_research.git"
REPOSITORY_ROOT = "/content/phd_research"
KAGGLE_DATASET_HANDLE = "sanelehlabisa/abnormal-activities-dataset"
KAGGLE_DATASET_FOLDER = "abnormal-activities-dataset"
KAGGLE_DOWNLOAD_ROOT = "/content/datasets"
DATASET_DIR = "/content/datasets/abnormal-activities-dataset"
RUNS_DIR = "/content/runs"  # Temporary; archive important runs before disconnecting.
SPLIT_MANIFEST_PATH = ""
EXPERIMENT_CONFIG_PATH = ""
CANDIDATE_MANIFEST_PATH = ""

# Preliminary validation-screen candidate; not a final or optimal model.
SELECTED_LAYERS = [(8, 3, 3), (8, 3, 3), (8, 3, 3)]
SEED = 42
CONFIRMATION_SEED = 42
SCREEN_SEQUENCE_LENGTH = 16
SCREEN_HEIGHT = 32
SCREEN_WIDTH = 32
TRAIN_SEQUENCE_LENGTH = 32
TRAIN_HEIGHT = 64
TRAIN_WIDTH = 64
BATCH_SIZE = 8
EPOCHS = 64
EARLY_STOPPING_PATIENCE = 10
NUM_WORKERS = 2

# Optional existing evidence or checkpoint paths.
SUMMARY_PATH = ""
CHECKPOINT_PATH = ""
RESUME_CHECKPOINT_PATH = ""
ARCHIVE_SOURCE_DIR = ""
ARCHIVE_OUTPUT_DIR = ""

# Every action is opt-in. Change only the stage you are ready to run.
CLONE_REPOSITORY = False
INSTALL_MISSING_PACKAGES = False
DOWNLOAD_AAD_FROM_KAGGLE = False
RUN_PATH_VALIDATION = False
RUN_DATASET_PREVIEW = False
RUN_MODEL_SMOKE = False
RUN_CONFIG_INSPECTION = False
RUN_ARCHITECTURE_SCREEN = False
RUN_BASELINE_CONFIRMATION = False
RUN_SELECTED_TRAINING = False
RUN_RESULT_INSPECTION = False
UNLOCK_TEST_EVALUATION = False
RUN_TEST_EVALUATION = False
RUN_ARCHIVE = False

## Portable helpers

Commands are printed before execution and use the notebook kernel's Python.
Missing paths and failed subprocesses stop the current cell with a clear error.

In [ ]:
import importlib.util
import json
import os
import platform
import shlex
import shutil
import subprocess
import sys


def run_command(command: list[str], cwd: Path | None = None) -> None:
    print("$", shlex.join(str(part) for part in command), flush=True)
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)


def cloud_platform() -> str:
    if "google.colab" in sys.modules:
        return "colab"
    if Path("/kaggle/working").is_dir():
        return "kaggle"
    return "local"


def default_clone_root() -> Path:
    if cloud_platform() == "colab":
        return Path("/content/phd_research")
    if cloud_platform() == "kaggle":
        return Path("/kaggle/working/phd_research")
    return Path.cwd() / "phd_research"


def find_repository_root() -> Path:
    marker = Path("papers/001-journal-abnormal-activity-recognition/implementation/src")
    candidates = []
    if REPOSITORY_ROOT.strip():
        candidates.append(Path(REPOSITORY_ROOT).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents, default_clone_root()])
    for candidate in candidates:
        resolved = candidate.resolve()
        if (resolved / marker).is_dir():
            return resolved
    raise FileNotFoundError(
        "Repository not found. Set REPOSITORY_ROOT or enable CLONE_REPOSITORY."
    )


def implementation_root() -> Path:
    return find_repository_root() / "papers/001-journal-abnormal-activity-recognition/implementation"


def required_path(value: str, label: str, kind: str = "file") -> Path:
    if not value.strip():
        raise ValueError(f"Set {label} in the parameter cell first.")
    path = Path(value).expanduser().resolve()
    exists = path.is_dir() if kind == "directory" else path.is_file()
    if not exists:
        raise FileNotFoundError(f"{label} does not exist: {path}")
    return path


def configured_paths(require_dataset: bool = False) -> dict[str, Path]:
    implementation = implementation_root()
    dataset = (
        required_path(DATASET_DIR, "DATASET_DIR", "directory")
        if require_dataset
        else Path(DATASET_DIR).expanduser().resolve()
        if DATASET_DIR.strip()
        else Path()
    )
    return {
        "implementation": implementation,
        "dataset": dataset,
        "runs": (
            Path(RUNS_DIR).expanduser().resolve()
            if RUNS_DIR.strip()
            else implementation / "runs"
        ),
        "split": (
            Path(SPLIT_MANIFEST_PATH).expanduser().resolve()
            if SPLIT_MANIFEST_PATH.strip()
            else implementation / "splits/abnormal-activities-dataset_seed42.json"
        ),
        "config": (
            Path(EXPERIMENT_CONFIG_PATH).expanduser().resolve()
            if EXPERIMENT_CONFIG_PATH.strip()
            else implementation / "configs/aad_screening_reference.json"
        ),
        "candidates": (
            Path(CANDIDATE_MANIFEST_PATH).expanduser().resolve()
            if CANDIDATE_MANIFEST_PATH.strip()
            else implementation / "configs/aad_architecture_candidates.json"
        ),
    }


def append_layers(command: list[str]) -> list[str]:
    for filters, kernel_height, kernel_width in SELECTED_LAYERS:
        command.extend(
            ["--convlstm-layer", str(filters), str(kernel_height), str(kernel_width)]
        )
    return command

## Optional repository setup

Uploading this notebook does not automatically clone or update the repository.
Enable cloning only when the repository is absent. Existing checkouts are never
pulled, reset, or overwritten.

In [ ]:
if CLONE_REPOSITORY:
    clone_root = (
        Path(REPOSITORY_ROOT).expanduser().resolve()
        if REPOSITORY_ROOT.strip()
        else default_clone_root()
    )
    if clone_root.exists():
        raise FileExistsError(f"Refusing to overwrite existing path: {clone_root}")
    clone_root.parent.mkdir(parents=True, exist_ok=True)
    run_command(["git", "clone", REPOSITORY_URL, str(clone_root)], cwd=clone_root.parent)
else:
    print("Repository clone disabled.")

## Environment and GPU report

Colab virtual machines are temporary, so use mounted persistent storage for
important runs. On Kaggle, keep datasets under attached input paths and write
runs or archives under a writable working/output path. Selecting a GPU runtime
does not guarantee that code uses it; confirm CUDA below.

In [ ]:
print("Platform:", cloud_platform())
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("System:", platform.platform())
print("Working directory:", Path.cwd())
print("Free disk (GB):", round(shutil.disk_usage(Path.cwd()).free / 1024**3, 2))

if importlib.util.find_spec("torch") is None:
    print("PyTorch is missing. Choose a supported accelerator runtime before training.")
else:
    import torch
    import torchvision

    print("PyTorch:", torch.__version__)
    print("Torchvision:", torchvision.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

## Optional missing dependencies

This installs only missing non-accelerator packages. It deliberately does not
replace `torch`, `torchvision`, or `torchaudio` supplied by the cloud runtime.
Keep the printed package versions with your run metadata.

In [ ]:
cloud_packages = {
    "av": "av==16.1.0",
    "cv2": "opencv-python",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "pillow",
    "kagglehub": "kagglehub",
    "sklearn": "scikit-learn",
    "torchmetrics": "torchmetrics==1.8.2",
    "tqdm": "tqdm",
}
missing_packages = [
    package
    for module, package in cloud_packages.items()
    if importlib.util.find_spec(module) is None
]
print("Missing packages:", missing_packages or "none")

if INSTALL_MISSING_PACKAGES:
    if importlib.util.find_spec("torch") is None:
        raise RuntimeError("PyTorch is missing; select a cloud GPU runtime first.")
    if missing_packages:
        run_command([sys.executable, "-m", "pip", "install", *missing_packages])
    else:
        print("Nothing to install.")
else:
    print("Dependency installation disabled.")

## Optional AAD download from Kaggle

Download the approved public dataset folder directly into the Colab path.
This network action is disabled by default. Public access normally needs no
credential; if Kaggle asks for authentication or consent, add
`KAGGLE_API_TOKEN` to Colab Secrets and rerun this cell. Never paste a token
into the notebook.


In [ ]:
if DOWNLOAD_AAD_FROM_KAGGLE:
    if importlib.util.find_spec("kagglehub") is None:
        raise RuntimeError("Install missing packages before downloading AAD.")
    import kagglehub

    download_root = Path(KAGGLE_DOWNLOAD_ROOT).expanduser().resolve()
    download_root.mkdir(parents=True, exist_ok=True)
    downloaded_path = Path(
        kagglehub.dataset_download(
            KAGGLE_DATASET_HANDLE,
            path=KAGGLE_DATASET_FOLDER,
            output_dir=str(download_root),
        )
    ).resolve()
    expected_path = Path(DATASET_DIR).expanduser().resolve()
    if downloaded_path != expected_path:
        raise RuntimeError(
            f"Downloaded dataset path {downloaded_path} does not match DATASET_DIR "
            f"{expected_path}; update DATASET_DIR before continuing."
        )
    print("AAD dataset:", downloaded_path)
else:
    print("Kaggle dataset download disabled.")


## Validate repository, dataset, configuration, and output paths

The Colab defaults point to the downloaded AAD class-folder directory and a
temporary runs directory. Change `RUNS_DIR` if persistent storage is mounted.
This stage creates only the configured runs directory.

In [ ]:
if RUN_PATH_VALIDATION:
    paths = configured_paths(require_dataset=True)
    required_path(str(paths["split"]), "SPLIT_MANIFEST_PATH")
    required_path(str(paths["config"]), "EXPERIMENT_CONFIG_PATH")
    required_path(str(paths["candidates"]), "CANDIDATE_MANIFEST_PATH")
    paths["runs"].mkdir(parents=True, exist_ok=True)
    for name, path in paths.items():
        print(f"{name}: {path}")
else:
    print("Path validation disabled.")

## Dataset preview

This calls `src.dataset` to write a few clean and clip-consistent augmented MP4
samples. It is a visual pipeline check, not research evidence.

In [ ]:
if RUN_DATASET_PREVIEW:
    paths = configured_paths(require_dataset=True)
    command = [
        sys.executable,
        "-m",
        "src.dataset",
        "--dataset_dir",
        str(paths["dataset"]),
        "--sequence_length",
        str(SCREEN_SEQUENCE_LENGTH),
        "--height",
        str(SCREEN_HEIGHT),
        "--width",
        str(SCREEN_WIDTH),
        "--num_samples",
        "2",
        "--fps",
        "8",
        "--seed",
        str(SEED),
        "--augment",
    ]
    run_command(command, cwd=paths["implementation"])
else:
    print("Dataset preview disabled.")

## Model smoke test

This calls `src.model` with tiny inputs and random weights to verify model
construction and video prediction output. It does not measure accuracy.

In [ ]:
if RUN_MODEL_SMOKE:
    paths = configured_paths(require_dataset=True)
    command = [
        sys.executable,
        "-m",
        "src.model",
        "--dataset-dir",
        str(paths["dataset"]),
        "--runs_dir",
        str(paths["runs"]),
        "--sequence-length",
        "2",
        "--height",
        "8",
        "--width",
        "8",
        "--seed",
        str(SEED),
        "--num-samples",
        "2",
    ]
    run_command(append_layers(command), cwd=paths["implementation"])
else:
    print("Model smoke test disabled.")

## Inspect the committed protocol and six candidates

These commands print configuration and model definitions without accessing AAD,
allocating models, or creating a research run.

In [ ]:
if RUN_CONFIG_INSPECTION:
    paths = configured_paths()
    required_path(str(paths["config"]), "EXPERIMENT_CONFIG_PATH")
    required_path(str(paths["candidates"]), "CANDIDATE_MANIFEST_PATH")
    run_command(
        [sys.executable, "-m", "src.train", "--config", str(paths["config"]), "--print-config"],
        cwd=paths["implementation"],
    )
    run_command(
        [
            sys.executable,
            "-m",
            "src.experiments",
            "--config",
            str(paths["config"]),
            "--candidates-config",
            str(paths["candidates"]),
            "--list-models",
        ],
        cwd=paths["implementation"],
    )
else:
    print("Configuration inspection disabled.")

## Optional architecture screen — expensive

This runs only the six explicit custom candidates using the committed screening
protocol. Do not rerun it merely to search for a more favourable result.

In [ ]:
if RUN_ARCHITECTURE_SCREEN:
    paths = configured_paths(require_dataset=True)
    command = [
        sys.executable,
        "-m",
        "src.experiments",
        "--config",
        str(paths["config"]),
        "--candidates-config",
        str(paths["candidates"]),
        "--dataset_dir",
        str(paths["dataset"]),
        "--runs_dir",
        str(paths["runs"]),
        "--split_manifest",
        str(paths["split"]),
    ]
    run_command(command, cwd=paths["implementation"])
else:
    print("Architecture screen disabled.")

## Optional audited baseline confirmation — expensive

Run this only after choosing the custom candidate from validation evidence. It
compares that explicit stack with the source-paper topology and the three
practical 3D-CNN baselines. Use one confirmation seed at a time and keep the
screening protocol fixed.

In [ ]:
if RUN_BASELINE_CONFIRMATION:
    paths = configured_paths(require_dataset=True)
    command = [
        sys.executable,
        "-m",
        "src.experiments",
        "--config",
        str(paths["config"]),
        "--dataset_dir",
        str(paths["dataset"]),
        "--runs_dir",
        str(paths["runs"]),
        "--split_manifest",
        str(paths["split"]),
        "--seed",
        str(CONFIRMATION_SEED),
    ]
    run_command(append_layers(command), cwd=paths["implementation"])
else:
    print("Baseline confirmation disabled.")

## Train the selected custom candidate — expensive

The defaults increase frame count and spatial size, so this is extended
selected-model training—not evidence from the original architecture-only
screen. The script still uses the committed split, validation-loss selection,
early stopping, and restored best checkpoint. Set `RESUME_CHECKPOINT_PATH` only
when resuming a compatible custom-model checkpoint.

In [ ]:
if RUN_SELECTED_TRAINING:
    paths = configured_paths(require_dataset=True)
    command = [
        sys.executable,
        "-m",
        "src.train",
        "--config",
        str(paths["config"]),
        "--dataset_dir",
        str(paths["dataset"]),
        "--runs_dir",
        str(paths["runs"]),
        "--split_manifest",
        str(paths["split"]),
        "--seed",
        str(SEED),
        "--sequence_length",
        str(TRAIN_SEQUENCE_LENGTH),
        "--height",
        str(TRAIN_HEIGHT),
        "--width",
        str(TRAIN_WIDTH),
        "--batch_size",
        str(BATCH_SIZE),
        "--epochs",
        str(EPOCHS),
        "--early_stopping_patience",
        str(EARLY_STOPPING_PATIENCE),
        "--num_workers",
        str(NUM_WORKERS),
    ]
    append_layers(command)
    if RESUME_CHECKPOINT_PATH.strip():
        checkpoint = required_path(RESUME_CHECKPOINT_PATH, "RESUME_CHECKPOINT_PATH")
        command.extend(["--checkpoint_path", str(checkpoint), "--resume"])
    run_command(command, cwd=paths["implementation"])
else:
    print("Selected-model training disabled.")

## Inspect validation evidence

Set `SUMMARY_PATH` for a specific experiment. If it is empty, this stage uses
the newest experiment summary under `RUNS_DIR`. It displays validation metrics
only and never opens a test checkpoint or test metrics.

In [ ]:
if RUN_RESULT_INSPECTION:
    paths = configured_paths()
    if SUMMARY_PATH.strip():
        summary_path = required_path(SUMMARY_PATH, "SUMMARY_PATH")
    else:
        summaries = sorted(paths["runs"].glob("experiments/*/summary.json"))
        if not summaries:
            raise FileNotFoundError(f"No experiment summaries under {paths['runs']}")
        summary_path = summaries[-1]
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    rows = []
    for result in summary.get("ranked", []):
        validation = result.get("validation_metrics", {})
        rows.append(
            {
                "name": result.get("name"),
                "validation_macro_f1": validation.get("macro_f1"),
                "validation_accuracy": validation.get("accuracy"),
                "validation_loss": validation.get("loss"),
                "parameters": result.get("num_params"),
                "selected_epoch": result.get("checkpoint_selection", {}).get("selected_epoch"),
            }
        )
    print("Summary:", summary_path)
    if (
        importlib.util.find_spec("pandas") is not None
        and importlib.util.find_spec("IPython") is not None
    ):
        import pandas as pd
        from IPython.display import display

        display(pd.DataFrame(rows))
    else:
        print(json.dumps(rows, indent=2))

    ranked = summary.get("ranked", [])
    if ranked:
        selected = ranked[0]
        print("Top validation-ranked model:", selected.get("name"))
        print("Selected checkpoint:", selected.get("selected_checkpoint"))
        history_value = selected.get("history")
        confusion_value = selected.get("validation_confusion_matrix", {}).get("png")

        def resolve_artifact(value: str | None) -> Path | None:
            if not value:
                return None
            candidate = Path(value).expanduser()
            choices = [candidate, paths["implementation"] / candidate]
            return next((item.resolve() for item in choices if item.is_file()), None)

        history_path = resolve_artifact(history_value)
        if history_path:
            history = json.loads(history_path.read_text(encoding="utf-8"))
            print("History:", history_path)
            print("Selection:", json.dumps(history.get("selection", {}), indent=2))
        else:
            print("Validation history was not found locally.")

        confusion_path = resolve_artifact(confusion_value)
        if confusion_path:
            print("Validation confusion matrix:", confusion_path)
            if importlib.util.find_spec("IPython") is not None:
                from IPython.display import Image, display

                display(Image(filename=str(confusion_path)))
            else:
                print("Open the PNG path above in your notebook interface.")
        else:
            print("Validation confusion-matrix image was not found locally.")
else:
    print("Result inspection disabled.")

## Final frozen-checkpoint test — deliberately locked

This is not a model-selection stage. Set an explicit custom-model checkpoint,
confirm its architecture and input dimensions above, then set **both**
`UNLOCK_TEST_EVALUATION` and `RUN_TEST_EVALUATION` to `True`. Evaluate one
frozen checkpoint once; never loop over candidates using test results.

In [ ]:
if RUN_TEST_EVALUATION:
    if not UNLOCK_TEST_EVALUATION:
        raise PermissionError("Set UNLOCK_TEST_EVALUATION=True only after selection is frozen.")
    paths = configured_paths(require_dataset=True)
    checkpoint = required_path(CHECKPOINT_PATH, "CHECKPOINT_PATH")
    command = [
        sys.executable,
        "-m",
        "src.evaluate",
        "--dataset_dir",
        str(paths["dataset"]),
        "--checkpoint_path",
        str(checkpoint),
        "--runs_dir",
        str(paths["runs"]),
        "--split_manifest",
        str(paths["split"]),
        "--seed",
        str(SEED),
        "--train_ratio",
        "0.7",
        "--val_ratio",
        "0.15",
        "--batch_size",
        str(BATCH_SIZE),
        "--sequence_length",
        str(TRAIN_SEQUENCE_LENGTH),
        "--height",
        str(TRAIN_HEIGHT),
        "--width",
        str(TRAIN_WIDTH),
        "--num_workers",
        str(NUM_WORKERS),
        "--pin_memory",
        "--num_samples",
        "8",
    ]
    run_command(append_layers(command), cwd=paths["implementation"])
else:
    print("Final test evaluation disabled and locked.")

## Archive one chosen run

Set an explicit run directory and persistent output directory. Archiving is
disabled by default and refuses to overwrite an existing ZIP file. On Colab,
point the output to mounted Drive; on Kaggle, use a writable working/output
directory and save the notebook version when finished.

In [ ]:
if RUN_ARCHIVE:
    source = required_path(ARCHIVE_SOURCE_DIR, "ARCHIVE_SOURCE_DIR", "directory")
    if not ARCHIVE_OUTPUT_DIR.strip():
        raise ValueError("Set ARCHIVE_OUTPUT_DIR to persistent writable storage.")
    output_dir = Path(ARCHIVE_OUTPUT_DIR).expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    archive_base = output_dir / source.name
    archive_path = archive_base.with_suffix(".zip")
    if archive_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing archive: {archive_path}")
    created = shutil.make_archive(str(archive_base), "zip", root_dir=source.parent, base_dir=source.name)
    print("Archive:", created)
else:
    print("Run archive disabled.")

## Handoff checklist

- Keep the exact `run.json`, resolved configuration, split hash, histories,
  selected checkpoint, metrics, confusion matrix, and environment versions.
- If a Colab session may end, archive or write runs to mounted persistent
  storage before disconnecting.
- Do not compare the extended 32-frame, 64×64 training directly against the
  16-frame, 32×32 screen as an architecture-only result.
- Run the second confirmation seed before making a stability claim.
- Keep test results out of architecture selection and ablation decisions.